# 03 — Empirical uncertainty characterization

## Question

What is the empirical distribution of (Δd, ΔE) in the ACN-Data Caltech site, on the calibration window?

## Why this test exists

The robust QUBO (F1) and ADOPT (F2) depend on the empirical joint distribution of behavioral uncertainty. If ΔE and Δd are nearly degenerate (e.g., ΔE ≈ 0 for all sessions), the robust QUBO is equivalent to the deterministic QUBO. If they are heavily skewed (e.g., long early-departure tail), the F1/F2 schedules will differ materially from F0.

## Method

For each session in the calibration window, compute:

- `ΔE = E_delivered − E_requested` (kWh; <0 = unmet demand; >0 = over-  delivery). Sign convention is frozen at Stage 1 §4.2 and corrected   in Stage 6 Part A.
- `Δd = d_requested − d_actual` (minutes; >0 = early; <0 = late).
  Sign convention is frozen at Stage 1 §4.2.

The data source is the live ACN-Data API (token-gated). The placeholder is a synthetic mixture model used only for **infrastructure validation** while the token is unavailable. The placeholder is explicitly labeled `source="placeholder"` on every sample and is **never** reported as a research result.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage5.uncertainty import (placeholder_uncertainty,
                                   distribution_stats, sign_breakdown,
                                   joint_dependence)
import numpy as np

samples, status = placeholder_uncertainty(seed=20260829)
print(f"Placeholder source: {status['source']}")
print(f"Placeholder n_records: {status['n_records']}")
print(f"Placeholder n_valid:   {status['n_valid']}")
print()
print('NOTE: this is the PLACEHOLDER distribution for infrastructure validation.')
print('It is NOT the ACN-Data distribution. The real distribution comes from')
print('the live API in notebook 10 once the token is supplied.')
print()
cal = [s for s in samples if s.calibration]
dd = np.array([s.delta_d_minutes for s in cal if s.delta_d_minutes is not None])
de = np.array([s.delta_e_kwh    for s in cal if s.delta_e_kwh    is not None])
print(f"Placeholder calibration Δd stats: {distribution_stats(dd)}")
print()
print(f"Placeholder calibration ΔE stats: {distribution_stats(de)}")
print()
print(f"Placeholder calibration Δd sign breakdown: {sign_breakdown(dd)}")
print(f"Placeholder calibration ΔE sign breakdown: {sign_breakdown(de)}")
print()
print(f"Placeholder joint dependence: {joint_dependence(dd, de)}")


## Result (placeholder distribution shown for infrastructure validation)

The placeholder is a 50/30/20 mixture for Δd (zero-mean Gaussian, negative-skewed Gaussian, point mass at 0) and a 70/25/5 mixture for ΔE (point mass at 0, small Gaussian, large negative Gaussian). This shape was chosen to roughly resemble prior literature on ACN-Data behavior. It is **not** the Stage 1 uncertainty model and is **not** the ACN-Data distribution.

## Interpretation

The placeholder is useful for one thing: it exercises the k-means scenario engine, the ADOPT γ calculator, and the robust QUBO construction without requiring the live API. It has revealed important properties of the scenario-averaged formulation — for example, the F1/F2 placeholder result (where F0 dominates F1/F2) is a property of the placeholder's K=8 cluster distribution, not a fundamental flaw in the formulation (see notebook 08 and Stage 7 §1).

## Limitations

- The placeholder is **not the real distribution**. Do not extrapolate   any conclusion from the placeholder to ACN-Data.
- The real ΔE/Δd distributions (from the live API) are reported in   notebook 10 once the token is supplied.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
